In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/store-sales-time-series-forecasting/oil.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/sample_submission.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/holidays_events.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/stores.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/train.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/test.csv
/kaggle/input/competitions/store-sales-time-series-forecasting/transactions.csv


In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import xgboost as xgb
from sklearn.linear_model import Ridge
from sklearn.preprocessing import LabelEncoder
from statsmodels.tsa.deterministic import DeterministicProcess, CalendarFourier
import warnings
import os
import time

warnings.filterwarnings('ignore')

# 1. Load Data
def load_data():
    print("[1/7] Loading data...")
    base_path = '/kaggle/input/store-sales-time-series-forecasting/'
    if not os.path.exists(base_path + 'train.csv'):
        base_path = '/kaggle/input/competitions/store-sales-time-series-forecasting/'
        
    train = pd.read_csv(base_path + 'train.csv', parse_dates=['date'])
    test = pd.read_csv(base_path + 'test.csv', parse_dates=['date'])
    stores = pd.read_csv(base_path + 'stores.csv')
    oil = pd.read_csv(base_path + 'oil.csv', parse_dates=['date'])
    holidays = pd.read_csv(base_path + 'holidays_events.csv', parse_dates=['date'])
    
    # Filter for efficiency (2017 only)
    train = train[train['date'] >= '2017-01-01']
    return train, test, stores, oil, holidays

# 2. Feature Engineering
def prepare_features(train, test, stores, oil, holidays):
    print("[2/7] Preparing features...")
    
    # Pre-rename to avoid KeyError
    stores = stores.rename(columns={'type': 'store_type'})
    holidays = holidays.rename(columns={'type': 'holiday_type'})
    
    # Oil: Interpolate and add lags
    oil = oil.set_index('date').resample('D').mean().interpolate().reset_index()
    oil['oil_lags_1'] = oil['dcoilwtico'].shift(1)
    oil['oil_rolling_7'] = oil['dcoilwtico'].rolling(7).mean()
    
    # Holidays: Aggressive deduplication
    holidays = holidays[holidays['transferred'] == False]
    nat_hol = holidays[holidays['locale'] == 'National'].drop_duplicates('date')
    reg_hol = holidays[holidays['locale'] == 'Regional'].rename(columns={'locale_name': 'state'}).drop_duplicates(['date', 'state'])
    loc_hol = holidays[holidays['locale'] == 'Local'].rename(columns={'locale_name': 'city'}).drop_duplicates(['date', 'city'])
    
    # Combine train and test for feature engineering
    df = pd.concat([train, test], axis=0)
    df['date'] = pd.to_datetime(df['date'])
    
    # Merge
    df = df.merge(stores, on='store_nbr', how='left')
    df = df.merge(oil, on='date', how='left')
    df = df.merge(nat_hol[['date', 'holiday_type']], on='date', how='left').rename(columns={'holiday_type': 'nat_hol'})
    df = df.merge(reg_hol[['date', 'state', 'holiday_type']], on=['date', 'state'], how='left').rename(columns={'holiday_type': 'reg_hol'})
    df = df.merge(loc_hol[['date', 'city', 'holiday_type']], on=['date', 'city'], how='left').rename(columns={'holiday_type': 'loc_hol'})
    
    # Time features
    df['day'] = df['date'].dt.day
    df['month'] = df['date'].dt.month
    df['dayofweek'] = df['date'].dt.dayofweek
    df['wages_day'] = ((df['day'] == 15) | (df['date'].dt.is_month_end)).astype(int)
    
    # Lags (Safe lags >= 16 days to avoid leakage)
    print("  Creating safe lags...")
    for lag in [16, 21, 30]:
        df[f'lag_{lag}'] = df.groupby(['store_nbr', 'family'])['sales'].transform(lambda x: x.shift(lag))
        df[f'promo_lag_{lag}'] = df.groupby(['store_nbr', 'family'])['onpromotion'].transform(lambda x: x.shift(lag))
    
    # Label Encoding
    le = LabelEncoder()
    for col in ['family', 'city', 'state', 'store_type', 'cluster', 'nat_hol', 'reg_hol', 'loc_hol']:
        df[col] = le.fit_transform(df[col].astype(str))
        
    return df

# 3. Hybrid Model Class
class BoostedHybrid:
    def __init__(self, model_1, model_2, model_3):
        self.model_1 = model_1 # Linear (Trend + Seasonality)
        self.model_2 = model_2 # LightGBM (Residuals)
        self.model_3 = model_3 # XGBoost (Residuals)
        self.y_columns = None
        
    def fit(self, X_1, X_2, y):
        # Fit Linear Trend
        self.model_1.fit(X_1, y)
        self.y_columns = y.columns
        
        # Calculate Residuals
        y_fit = pd.DataFrame(self.model_1.predict(X_1), index=X_1.index, columns=y.columns)
        y_resid = y - y_fit
        y_resid = y_resid.stack(['store_nbr', 'family'])
        
        # Fit Residual Models
        print("  Fitting LightGBM on residuals...")
        self.model_2.fit(X_2, y_resid)
        print("  Fitting XGBoost on residuals...")
        self.model_3.fit(X_2, y_resid)
        
    def predict(self, X_1, X_2):
        # Linear Predict
        y_pred = pd.DataFrame(self.model_1.predict(X_1), index=X_1.index, columns=self.y_columns)
        y_pred = y_pred.stack(['store_nbr', 'family'])
        
        # Residual Predict (Ensemble)
        resid_lgb = self.model_2.predict(X_2)
        resid_xgb = self.model_3.predict(X_2)
        y_pred += (resid_lgb * 0.6 + resid_xgb * 0.4)
        
        return y_pred.unstack(['store_nbr', 'family'])

# 4. Main Execution
def main():
    train, test, stores, oil, holidays = load_data()
    df = prepare_features(train, test, stores, oil, holidays)
    
    # --- Stage 1: Trend & Seasonality (Deterministic Process) ---
    print("[3/7] Modeling Trend & Seasonality...")
    y = df[df['date'] < '2017-08-16'].set_index(['date', 'store_nbr', 'family'])['sales'].unstack(['store_nbr', 'family']).fillna(0)
    y.index = pd.to_datetime(y.index)
    y = np.log1p(y.asfreq('D').fillna(0))
    
    fourier = CalendarFourier(freq="A", order=4)
    dp = DeterministicProcess(index=y.index, constant=True, order=1, additional_terms=[fourier], drop=True)
    X_1 = dp.in_sample()
    X_test_1 = dp.out_of_sample(steps=16)
    
    # --- Stage 2: Residuals (LightGBM + XGBoost) ---
    print("[4/7] Preparing residual features...")
    features_2 = [c for c in df.columns if c not in ['id', 'date', 'sales', 'dcoilwtico']]
    
    # Ensure perfect alignment for residual modeling
    df_train = df[df['date'] < '2017-08-16'].sort_values(['date', 'store_nbr', 'family'])
    df_test = df[df['date'] >= '2017-08-16'].sort_values(['date', 'store_nbr', 'family'])
    
    # Re-align features to the padded y index
    X_2_aligned = df_train.set_index(['date', 'store_nbr', 'family']).reindex(
        pd.MultiIndex.from_product([y.index, y.columns.get_level_values(0).unique(), y.columns.get_level_values(1).unique()], names=['date', 'store_nbr', 'family'])
    ).reset_index()
    X_2 = X_2_aligned[features_2].fillna(0)
    X_test_2 = df_test[features_2].fillna(0)
    
    # Models
    model_1 = Ridge(alpha=0.5)
    model_2 = lgb.LGBMRegressor(n_estimators=2000, learning_rate=0.015, num_leaves=127, random_state=42, verbosity=-1, device="gpu")
    model_3 = xgb.XGBRegressor(n_estimators=1500, learning_rate=0.02, max_depth=8, tree_method='hist', device='cuda', random_state=42)
    
    print("[5/7] Training Hybrid Ensemble...")
    hybrid = BoostedHybrid(model_1, model_2, model_3)
    hybrid.fit(X_1, X_2, y)
    
    print("[6/7] Generating predictions...")
    y_pred = hybrid.predict(X_test_1, X_test_2)
    
    # Final formatting
    print("[7/7] Finalizing submission...")
    # Explicitly name columns to avoid KeyError: 'date'
    y_submit = y_pred.stack(['store_nbr', 'family']).reset_index()
    y_submit.columns = ['date', 'store_nbr', 'family', 'sales']
    y_submit['sales'] = np.expm1(y_submit['sales']).clip(0, None)
    
    # Ensure identical types for merge
    test['date'] = pd.to_datetime(test['date'])
    y_submit['date'] = pd.to_datetime(y_submit['date'])
    test['family'] = test['family'].astype(str)
    y_submit['family'] = y_submit['family'].astype(str)
    test['store_nbr'] = test['store_nbr'].astype(int)
    y_submit['store_nbr'] = y_submit['store_nbr'].astype(int)
    
    submission = test.merge(y_submit, on=['date', 'store_nbr', 'family'], how='left')
    
    # Set Jan 1st to zero (stores closed)
    submission.loc[submission['date'].dt.dayofyear == 1, 'sales'] = 0
    
    submission[['id', 'sales']].to_csv('submission.csv', index=False)
    print("Success! Submission saved to submission.csv")

if __name__ == "__main__":
    main()

[1/7] Loading data...
[2/7] Preparing features...
  Creating safe lags...
[3/7] Modeling Trend & Seasonality...
[4/7] Preparing residual features...
[5/7] Training Hybrid Ensemble...
  Fitting LightGBM on residuals...


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


  Fitting XGBoost on residuals...
[6/7] Generating predictions...
[7/7] Finalizing submission...
Success! Submission saved to submission.csv
